# 01. DART 공시 데이터 수집

한국 시가총액 상위 10개 기업의 사업·반기·분기보고서를 DART에서 다운로드합니다.

**소요 시간**: 약 10-15분 (네트워크 속도에 따라)

## 1.1 환경 설정

In [ ]:
# 필요 패키지 설치
!pip install -q requests beautifulsoup4 lxml tqdm

In [ ]:
import os, sys
from google.colab import userdata, drive

# Drive 마운트 (백업용, 선택)
# drive.mount('/content/drive')

# 환경변수 설정 (Colab Secrets 사용)
try:
    os.environ['DART_API_KEY'] = userdata.get('DART_API_KEY')
    print("✅ DART API 키 로드")
except Exception as e:
    print(" Colab Secrets에 DART_API_KEY 등록 필요")
    print("   https://opendart.fss.or.kr 에서 무료 발급")
    raise

## 1.2 대상 기업 (한국 시가총액 상위 10개)

In [ ]:
# 기업 코드 (DART 고유번호)
TARGET_COMPANIES = {
    '삼성전자':        '00126380',
    'SK하이닉스':       '00164779',
    'LG에너지솔루션':    '01566260',
    '삼성바이오로직스':  '00877059',
    '현대자동차':       '00164742',
    '기아':            '00106641',
    'NAVER':          '00266961',
    'POSCO홀딩스':     '00434003',
    'LG화학':          '00356361',
    '셀트리온':         '00421045',
}

print(f"대상 기업: {len(TARGET_COMPANIES)}개")
for name, code in TARGET_COMPANIES.items():
    print(f"  - {name}: {code}")

## 1.3 보고서 다운로드 (XBRL 본문)

**중요**: DART의 본문 XBRL 의무화는 2023년 11월 14일 제출분(2023년 3분기 보고서)부터 적용됩니다. 그 이전 보고서는 XBRL 본문이 없습니다.

In [ ]:
# src/data/data_crawl.py 사용
# 또는 직접 호출:

import sys
sys.path.insert(0, '/content')

# 저장소 클론
!git clone https://github.com/<yjr2003>/AI-Stock-Analysis-Assistant.git /content/repo 2>/dev/null || echo "이미 존재"
sys.path.insert(0, '/content/repo')

# 크롤러 import
from src.data.data_crawl import crawl_company_reports

# 출력 디렉토리
RAW_DIR = '/content/data/raw'
os.makedirs(RAW_DIR, exist_ok=True)

# 각 기업 보고서 다운로드 (2023.3Q ~ 2025.사업보고서)
for name, corp_code in TARGET_COMPANIES.items():
    print(f"\n=== {name} ({corp_code}) ===")
    crawl_company_reports(
        corp_code=corp_code,
        corp_name=name,
        out_dir=RAW_DIR,
        start_date='20231114',  # XBRL 의무화 시작일
        end_date='20260430',
    )

print("\n✅ 데이터 수집 완료")

## 1.4 결과 확인

In [ ]:
import os

total_files = 0
for company_dir in sorted(os.listdir(RAW_DIR)):
    company_path = os.path.join(RAW_DIR, company_dir)
    if os.path.isdir(company_path):
        files = [f for f in os.listdir(company_path) if f.endswith('.xml') or f.endswith('.zip')]
        print(f"  {company_dir}: {len(files)}개 파일")
        total_files += len(files)

print(f"\n총 보고서 파일: {total_files}개")
print(f"저장 위치: {RAW_DIR}")

## 1.5 다음 단계

수집된 XBRL 본문을 파싱하여 사실(Fact) DB로 변환합니다.

→ **02_xbrl_parsing.ipynb** 로 진행